In [0]:
# =============================================================
# gold_reader.py
# Layer     : Gold
# Purpose   : Reads from Silver Delta only.
#             Provides slim, pre-selected DataFrames to avoid
#             shuffling unnecessary columns through joins.
#
# STRICT RULE: No writes. No aggregations. No business logic.
#              If you see groupBy here → wrong file.
# =============================================================

from pyspark.sql.functions import col


def _silver(table_name: str):
    """Base read from Silver Delta."""
    path = abfss("silver", table_name)
    return spark.read.format("delta").load(path)


# ── Dimension readers — slim column selection ─────────────────────────────────

def read_owners():
    return _silver("owners").select(
        "owner_id", "owner_name", "city", "state", "country"
    )

def read_pets():
    return _silver("pets").select(
        "pet_id", "owner_id", "pet_name", "species", "breed", "gender", "birth_date"
    )

def read_products():
    return _silver("products").select(
        "product_id", "product_name", "device_category", "msrp", "manufacturing_cost"
    )

def read_firmware():
    return _silver("firmware").select(
        "firmware_id", "firmware_version", "release_date", "release_type"
    )

def read_batches():
    return _silver("batches").select(
        "batch_id", "factory_name", "factory_location", "production_date"
    )

def read_devices():
    return _silver("devices").select(
        "device_id", "pet_id", "product_id", "batch_id", "firmware_id",
        "purchase_date", "warranty_end_date"
    )


# ── Fact readers — all columns needed for aggregation ────────────────────────

def read_activity_snapshots():
    return _silver("activity_snapshots").select(
        "device_id", "pet_id", "snapshot_timestamp",
        "steps", "distance_km", "active_minutes"
    )

def read_health_snapshots():
    return _silver("health_snapshots").select(
        "device_id", "pet_id", "snapshot_timestamp",
        "heart_rate", "pulse_rate", "body_temperature"
    )

def read_feeding_events():
    return _silver("feeding_events").select(
        "device_id", "pet_id", "event_timestamp", "food_dispensed_grams"
    )

def read_hydration_events():
    return _silver("hydration_events").select(
        "device_id", "pet_id", "event_timestamp", "water_consumed_ml"
    )

def read_device_snapshots():
    return _silver("device_snapshots").select(
        "device_id", "snapshot_timestamp",
        "battery_pct", "signal_strength", "temperature_c"
    )

def read_device_failures():
    return _silver("device_failures").select(
        "device_id", "failure_id", "failure_timestamp",
        "failure_type", "downtime_minutes", "severity_score"
    )

def read_firmware_deployments():
    return _silver("firmware_deployments").select(
        "device_id", "firmware_id", "deployment_timestamp",
        "deployment_status", "rollback_flag", "deployment_duration_seconds"
    )

def read_sales():
    return _silver("sales").select(
        "sale_id", "device_id", "owner_id", "sale_date",
        "sale_price", "discount_amount", "sales_channel", "city", "state"
    )

print("[gold_reader] Loaded. Silver readers ready for all 14 tables.")